# RAG Playground: Modüler Bileşenlerle Asistan İnşası

## RAG Mimari Akışı

Doküman Yükleme -> Text Extraction -> Chunking -> Embedding -> VectorDB'ye Ekleme -> Sorgu / Guardrails -> Intent -> Retrieval (Dense + BM25 -> RRF -> Reranker) -> Cevap Üretimi (LLM-as-a-Judge) -> Hafıza Güncelleme

---

## Adım Adım Kullanılacak Sınıflar ve Parametreleri

Aşağıda mimariyi oluşturan her bir yapboz parçasının görevi ve kurucu (`__init__`) parametreleri listelenmiştir. Detaylı teorik açıklamalar ve görselleştirmeler `notebooks/demo.ipynb` dosyasında yer almaktadır.

### 0. Temel LLM İstemcisi (`LLMFactory`)
Model API çağrılarını sağlayıcıdan (Azure OpenAI, OpenAI, Gemini) bağımsızlaştıran temel istemcidir.
- `provider` (`str`, varsayılan `settings.LLM_PROVIDER`): Kullanılacak LLM sağlayıcısı (`azure`, `openai`, `gemini`).
- `timeout` (`int`, varsayılan `60`): Model API istekleri için zaman aşımı süresi (saniye).
- `max_retries` (`int`, varsayılan `3`): Ağ hatası veya rate limit durumunda yeniden deneme sayısı.

### 1. Doküman Girişi ve Metin Çıkarma (`DocumentIO` & `DocumentExtractor`)
Ham PDF dokümanını okur ve yapılandırılmış Markdown metnine dönüştürür.
- **`DocumentIO`**:
  - `encoding` (`str`, varsayılan `'utf-8'`): Metin dosyaları için karakter kodlaması.
- **`DocumentExtractor`**:
  - `ocr_enabled` (`bool`, varsayılan `False`): Taranmış belgeler için OCR motorunu açar.
  - `force_backend_text` (`bool`, varsayılan `False`): Doğrudan backend metin çıkarımını zorlar.
  - `image_scale` (`float`, varsayılan `3.0`): Sayfa render görüntü ölçeği.
  - `ocr_factory` (`OCRFactory | None`, varsayılan `None`): OCR dil ve sayfa ayarları fabrikası.
  - `layout_enabled` (`bool`, varsayılan `True`): Başlık ve sayfa düzeni analizi yapar.
  - `table_enabled` (`bool`, varsayılan `True`): Tabloları satır/sütun yapısıyla Markdown'a çevirir.
  - `do_cell_matching` (`bool`, varsayılan `True`): Tablo hücre koordinat eşleştirmesini aktif eder.
  - `do_formula_enrichment` (`bool`, varsayılan `True`): Matematiksel formülleri LaTeX gösterimine çevirir.
  - `generate_picture_images` (`bool`, varsayılan `True`): Görsel ve tablo resimlerini ayıklar.

### 2. Akıllı Parçalama (`DocumentChunker`)
Metni başlık hiyerarşisi (`breadcrumb`), tablolar ve cümle embedding mesafelerine göre anlamsal parçalara böler.
- `embedder` (`EmbeddingService`, zorunlu): Cümleler arası anlamsal mesafeyi hesaplayan servis.
- `max_chunk_chars` (`int`, varsayılan `700`): Parçanın karakter üst hedefi.
- `min_chunk_chars` (`int`, varsayılan `260`): Küçük parçaların öncekiyle birleştirilme alt sınırı.
- `sentence_overlap_ratio` (`float`, varsayılan `0.2`): Önceki parçadan taşınacak cümle örtüşme oranı (%20).
- `semantic_chunking_break_percentage` (`float`, varsayılan `0.85`): Semantik kopma persentil eşiği.
- `semantic_chunking_min_dist` (`float`, varsayılan `0.2`): Bölme için gereken minimum kosinüs mesafesi ($1 - \text{CosineSimilarity}$).
- `semantic_chunking_buffer_size` (`int`, varsayılan `1`): Cümle vektörü için komşu penceresi (önceki + mevcut + sonraki).
- `semantic_chunking_min_sentences` (`int`, varsayılan `5`): Semantik bölme uygulanacak asgari cümle sayısı.
- `process_logging` (`bool`, varsayılan `False`): Parçalama loglarını konsola yazdırır.

### 3. Vektörleştirme (`LLMEmbeddingService`)
Metinleri çok boyutlu embedding vektörlerine dönüştürür.
- `client` (`LLMClient`, zorunlu): Embedding API çağrısını yapacak istemci.
- `deployment_name` (`str`, zorunlu): Kullanılacak embedding model adı (`settings.llm_models.embedding`).
- `embed_batch_size` (`int`, varsayılan `96`): Tek istekte gönderilecek maksimum metin sayısı.
- `timeout` (`int`, varsayılan `60`): Zaman aşımı süresi (saniye).

### 4. Vektör Veritabanı ve Depolama (`AsyncQdrantVectorDB`)
Metinleri ve vektörleri Qdrant koleksiyonuna kaydeder; eşzamanlı olarak bellek içi BM25 indeksini yönetir.
- `embedder` (`EmbeddingService`, zorunlu): Arama ve kayıt sırasında metinleri vektöre dönüştürecek servis.
- `host` (`str`, varsayılan `settings.QDRANT_HOST`): Qdrant sunucu adresi (`localhost`).
- `port` (`int`, varsayılan `settings.QDRANT_PORT`): Qdrant port numarası (`6333`).
- `api_key` (`str | None`, varsayılan `None`): Vektör veritabanı erişim anahtarı.
- `bm25_k1` (`float`, varsayılan `1.5`): BM25 terim sıklığı doygunluk katsayısı.
- `bm25_b` (`float`, varsayılan `0.75`): BM25 doküman uzunluğu normalizasyon katsayısı.
- `upsert_batch_size` (`int`, varsayılan `128`): Qdrant'a tek seferde gönderilecek kayıt sayısı.

### 5. Hibrit Arama ve Yeniden Sıralama (`RetrievalService`)
Dense (vektör) ve Sparse (BM25) aramayı RRF ile birleştirir; ardından adayları LLM hakemiyle puanlayarak filtreler.
- `vector_db` (`VectorDB`, zorunlu): Hibrit aramanın yapılacağı veritabanı.
- `client` (`LLMClient`, zorunlu): Puanlama yapacak model istemcisi.
- `reranker_active` (`bool`, varsayılan `True`): LLM ile yeniden sıralamayı açar/kapatır.
- `reranker` (`str`, varsayılan `settings.llm_models.reranker`): Puanlayıcı deployment adı (`gpt-4o-mini`).
- `reranker_system_prompt` (`str`, varsayılan `RERANK_SYSTEM_PROMPT`): Hakemin puanlama kriterleri.
- `temperature` (`float`, varsayılan `0.0`): Puanlama tutarlılığı için sıcaklık.
- `rerank_passage_chars` (`int`, varsayılan `700`): LLM'e gönderilecek maksimum pasaj karakteri.
- `min_rerank_score` (`int`, varsayılan `3`): Bu puanın altındaki parçalar elenir (0-10 arası).
- `response_format` (`dict`, varsayılan `LLM_RERANK_RESPONSE_FORMAT`): Yapılandırılmış JSON şeması.
- `timeout` (`int`, varsayılan `30`): Zaman aşımı süresi (saniye).

### 6. Güvenlik ve Çıktı Denetimi (`GuardrailsService`)
Girdide Prompt Injection engeller, PII (TCKN, IBAN, Kredi Kartı, Telefon) maskeler; çıktıda LLM-as-a-Judge ile halüsinasyon denetimi ve geri bildirim (feedback) döngüsü sağlar.
- `mode` (`Literal["strict", "lenient", "loose"]`, varsayılan `"strict"`): Maskeleme kapsamı katılığı.
- `regex_patterns` (`list[Pattern] | None`, varsayılan `None`): Özel regex maskeleme kuralları.
- `proprietary_dictionary` (`list[str] | None`, varsayılan `None`): Kuruma özel gizli sözcükler.
- `normalization_form` (`Literal["NFC", "NFD", "NFKC", "NFKD"]`, varsayılan `"NFKC"`): Unicode normalizasyonu.
- `llm_as_judge_enabled` (`bool`, varsayılan `True`): Çıktı denetimini açar/kapatır.
- `client` (`LLMClient | None`, varsayılan `None`): Hakem denetimi model istemcisi.
- `temperature` (`float | None`, varsayılan `0.0`): Hakem sıcaklık değeri.
- `guardrails_deployment` (`str | None`, varsayılan `settings.llm_models.guardrails`): Hakem model adı (`gpt-4o-mini`).
- `llm_as_judge_prompt` (`str | None`, varsayılan `LLM_AS_JUDGE_PROMPT`): Hakem talimatı.
- `response_format` (`dict | None`, varsayılan `LLM_GUARDRAILS_FORMAT`): JSON şeması.
- `timeout` (`int | None`, varsayılan `30`): Zaman aşımı süresi.

### 7. Niyet Sınıflandırma ve Sorgu Çözümleme (`IntentClassifier`)
Kullanıcı mesajının niyetini belirler (`knowledge_question`, `chitchat`, `needs_clarification`, `out_of_scope`) ve zamirleri çözümlenmiş bağımsız arama sorgusunu (`resolved_query`) üretir.
- `client` (`LLMClient`, zorunlu): Sınıflandırma yapacak LLM istemcisi.
- `system_prompt` (`str`, varsayılan `INTENT_SYSTEM_PROMPT`): Niyet sınıflandırma talimatı.
- `classifier_model` (`str`, varsayılan `settings.llm_models.intent`): Model deployment adı (`gpt-4o-mini`).
- `temperature` (`float`, varsayılan `0.0`): Sıcaklık.
- `response_format` (`dict`, varsayılan `LLM_INTENT_FORMAT`): JSON çıktı şeması.
- `timeout` (`int`, varsayılan `30`): Zaman aşımı süresi.

### 8. Konuşma ve Kullanıcı Belleği (`MemoryService`)
Kısa süreli konuşma geçmişini (Rolling Window) ve uzun süreli kullanıcı tercihlerini/olgularını (Semantic Memory) yönetir.
- `client` (`LLMClient`, zorunlu): Hafıza çıkarımı yapacak istemci.
- `vector_db` (`VectorDB`, zorunlu): Anıların saklanacağı veritabanı.
- `user_id` (`str`, zorunlu): Kullanıcı kimliği (ayrım için).
- `history_max_messages` (`int`, varsayılan `6`): Kısa süreli bellekte tutulacak maksimum mesaj çifti.
- `history_max_tokens` (`int`, varsayılan `2000`): Kısa süreli bellek maksimum token sınırı.
- `long_term_memory_batch_size` (`int`, varsayılan `10`): Uzun süreli aktarımı tetikleyen mesaj sayısı.
- `long_term_memory_min_score` (`float`, varsayılan `0.5`): Anı eşleşmesi için gereken benzerlik skoru.
- `memory_extraction_min_confidence` (`float`, varsayılan `0.7`): Bilginin hafızaya yazılması için asgari güven eşiği.
- `memory_extraction_model` (`str`, varsayılan `settings.llm_models.memory`): Bellek çıkarım modeli (`gpt-4o-mini`).
- `system_prompt` (`str`, varsayılan `MEMORY_SYSTEM_PROMPT`): Bellek çıkarım talimatı.
- `memory_response_format` (`dict`, varsayılan `LLM_MEMORY_RESPONSE_FORMAT`): JSON şeması.
- `timeout` (`int`, varsayılan `30`): Zaman aşımı süresi.

### 9. Asistan Orkestrasyonu (`AssistantService`)
Tüm bu modülleri bir araya getiren ana orkestratördür.
- `client` (`LLMClient`, zorunlu): Sohbet yanıtını üretecek model istemcisi.
- `intent_classifier` (`IntentClassifier`, zorunlu): Niyet belirleyici servis.
- `retriever` (`RetrievalService`, zorunlu): Kaynak dokümanları getiren servis.
- `guardrails` (`GuardrailsService | None`, varsayılan `None`): Güvenlik denetçisi.
- `memory_service` (`MemoryIService | None`, varsayılan `None`): Bellek servisi.
- `guardrails_enabled` (`bool`, varsayılan `True`): Guardrails ve LLM-as-a-Judge'ı açar/kapatır.
- `memory_enabled` (`bool`, varsayılan `True`): Bellek entegrasyonunu açar/kapatır.
- `model` (`str`, varsayılan `settings.llm_models.chat`): Yanıtı üretecek ana sohbet modeli (`gpt-5.4` / `gpt-4o`).
- `temperature` (`float`, varsayılan `0.3`): Üretim çeşitliliği (RAG için 0.2 - 0.4 önerilir).
- `system_prompt` (`str`, varsayılan `ASSISTANT_SYSTEM_PROMPT`): Asistanın davranış ve kaynak kuralları.
- `timeout` (`int`, varsayılan `30`): Zaman aşımı süresi.


In [ ]:
# ==============================================================================
# DOKÜMAN YÜKLEME WIDGET'I
#
# Nasıl Kullanılır:
# 1. Bu hücreyi çalıştırın ve 'Upload' butonuna tıklayarak bilgisayarınızdan bir PDF seçin.
# 2. Seçtiğiniz PDF otomatik olarak 'data/documents/raw_documents' dizinine kaydedilir.
# 3. 'saved_pdf_path' değişkeni seçilen PDF'in tam yolunu tutar; bu yolu kendi extractor kodunuzda kullanabilirsiniz.
# 4. Eğer dosya seçmezseniz sistem depodaki hazır 'AS19_Sayi-RAG-Sistemlerine-Giris.pdf' dosyasını varsayılan olarak atar.
# ==============================================================================

from pathlib import Path
import ipywidgets as widgets
from rag_playground.core.settings import PROJECT_ROOT
from rag_playground.ingestion.document_io import DocumentIO
from rag_playground.models.file_type_model import FileType

upload_widget = widgets.FileUpload(
    accept=str(FileType.PDF),
    multiple=False,
    description="PDF Seç"
)

document_io = DocumentIO(encoding="utf-8")
raw_dir = Path(str(PROJECT_ROOT / "data/documents/raw_documents"))

# Varsayılan dosya yolu
saved_pdf_path = raw_dir / "AS19_Sayi-RAG-Sistemlerine-Giris.pdf"

async def save_selected_file():
    global saved_pdf_path
    if upload_widget.value:
        uploaded_file = upload_widget.value[0]
        filename = Path(uploaded_file["name"]).stem
        content = bytes(uploaded_file["content"])
        saved_pdf_path = await document_io.save_document(
            content=content,
            filename=filename,
            save_dir=raw_dir,
            file_type=FileType.PDF
        )
        print(f"Yeni dosya yüklendi ve kaydedildi: {saved_pdf_path}")
    else:
        print(f"Dosya seçilmedi, varsayılan dosya kullanılıyor: {saved_pdf_path}")

display(upload_widget)
await save_selected_file()


In [ ]:
# ==============================================================================
# İNTERAKTİF KONUŞMA KUTUCUĞU (CHAT WIDGET'I)
#
# Nasıl Kullanılır:
# 1. Bir sonraki çalışma alanında (tuval) kendi asistanınızı 'chatbot' adıyla oluşturun.
# 2. Bu hücreyi çalıştırarak aşağıdaki konuşma arayüzünü açın.
# 3. Metin kutusuna sorunuzu yazıp 'Gönder' butonuna tıklayın veya Enter'a basın.
# 4. Asistanınızın yanıtı ve arka planda getirdiği kaynaklar (citations) anında ekrana yazdırılacaktır.
# ==============================================================================

import asyncio
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

text_input = widgets.Text(
    value='',
    placeholder='Sorunuzu buraya yazın (örn: RAG mimarisi nedir?)...',
    description='Soru:',
    layout=widgets.Layout(width='70%')
)

btn_send = widgets.Button(
    description='Gönder',
    button_style='primary',
    layout=widgets.Layout(width='14%')
)

btn_clear = widgets.Button(
    description='Temizle',
    button_style='warning',
    layout=widgets.Layout(width='14%')
)

chat_output = widgets.Output()

def handle_query(b=None):
    user_query = text_input.value.strip()
    if not user_query:
        return
    text_input.value = ''
    
    with chat_output:
        print(f"\nSoru: {user_query}")
        
        # Aşağıdaki çalışma alanında öğrencinin tanımlayacağı 'chatbot' nesnesini dinamik olarak alır
        active_chatbot = globals().get('chatbot')
        if active_chatbot is None:
            print("Hata: 'chatbot' nesnesi bulunamadı! Lütfen aşağıdaki çalışma alanında AssistantService nesnenizi 'chatbot' adıyla oluşturup çalıştırın.")
            return
        
        async def run_ask():
            try:
                response = await active_chatbot.answer(user_query)
                display(Markdown(f"**Yanıt:**\n{response.answer}"))
                if response.citations:
                    print(f"\nKullanılan Kaynaklar ({len(response.citations)} adet):")
                    for i, c in enumerate(response.citations, 1):
                        title = c.payload.get('doc_title', 'Doküman')
                        page = c.payload.get('page', 0)
                        score = getattr(c, 'rerank_score', None) or c.payload.get('score', '')
                        score_str = f" - Puan: {score}" if score else ""
                        print(f"  [{i}] {title} (Sayfa {page}){score_str}")
            except Exception as e:
                print(f"Hata oluştu: {e}")
        
        asyncio.create_task(run_ask())

def handle_clear(b=None):
    with chat_output:
        clear_output()

btn_send.on_click(handle_query)
text_input.on_submit(handle_query)
btn_clear.on_click(handle_clear)

chat_ui = widgets.VBox([
    widgets.HBox([text_input, btn_send, btn_clear]),
    chat_output
])
display(chat_ui)


In [ ]:
# ==============================================================================
# ÖĞRENCİ ÇALIŞMA ALANI: KENDİ ASİSTANINIZI İNŞA EDİN (TUVAL)
#
# Yukarıdaki parametre rehberine bakarak yapboz parçalarını burada birleştirin.
#
# Adım 1: LLMFactory ile temel 'client' nesnenizi oluşturun.
# Adım 2: DocumentIO, DocumentExtractor ve DocumentChunker nesnelerini kurun.
# Adım 3: 'saved_pdf_path' dosyasından extractor ile metni çıkarın ve chunker ile parçalayın.
# Adım 4: LLMEmbeddingService ve AsyncQdrantVectorDB nesnelerini oluşturup parçaları kaydedin.
# Adım 5: RetrievalService, GuardrailsService, IntentClassifier ve MemoryService nesnelerini tanımlayın.
# Adım 6: Tüm bu parçaları AssistantService altında 'chatbot' adıyla birleştirin.
# Adım 7: Parametrelerle (temperature, reranker_active, min_rerank_score, mode vb.) oynayın ve yukarıdaki sohbet widget'ında test edin!
# ==============================================================================

# Kendi kodlarınızı bu alana yazabilirsiniz:



